# JARVIS Power Coder - T4 Colab (Ollama Backend)

**Run a powerful ~8-10B unrestricted coder model on free Google Colab T4 GPU and expose a public URL for your local JARVIS Kali (or Mac) UI.**

This replaces the small 3B model on your old droplet with something much stronger for planning, coding PoCs, analyzing scans, and red team command generation.

### Quick Start
1. **Runtime → Change runtime type → GPU → T4** (important!)
2. Run cells in order.
3. Edit the `MODEL = "..."` line to pull your preferred 8-10B model (or paste any HF GGUF url supported by Ollama).
4. Start a tunnel (ngrok recommended or pinggy).
5. Copy the printed public https URL → paste into your local JARVIS UI **Settings → Ollama Base URL**.
6. Set Model Name to the tag you pulled (e.g. `dolphin-llama3:8b` or `jarvis-coder`).
7. Test connection and enjoy the much more powerful brain.

The notebook installs **zstd first**, GPU-related debs, starts Ollama cleanly, and is tuned to avoid OOM on T4's ~16 GB VRAM.

## 0. Confirm T4 GPU (free tier)
If this fails or shows CPU, go back to Runtime → Change runtime type and select **T4**.

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

## 1. Install zstd + GPU debs (required for Ollama + stability on Colab T4)
This matches common working recipes for Ollama on Colab. `zstd` is needed by the Ollama installer for extraction. GPU debs + LD_LIBRARY_PATH help the runtime see the T4 properly.

In [ ]:
print('Installing zstd and GPU / system packages...')
!sudo apt-get update -qq
!sudo apt-get install -y zstd pciutils

!echo 'debconf debconf/frontend select Noninteractive' | sudo debconf-set-selections
!sudo apt-get update -qq
!sudo apt-get install -y cuda-drivers || echo 'Note: cuda-drivers may be partially present in Colab runtime or can be skipped safely.'

import os
os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia:' + os.environ.get('LD_LIBRARY_PATH', '')
print('LD_LIBRARY_PATH set for NVIDIA libs.')
!echo 'zstd + GPU debs step complete.'

## 2. Install Ollama

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version || echo 'ollama binary check'

## 3. Start Ollama server in background (no OOM yet - just the daemon)

In [ ]:
import subprocess, time, os, signal

# Clean previous runs in this session
os.system('pkill -f "ollama serve" || true')
time.sleep(1)

logf = open('/content/ollama.log', 'w')
proc = subprocess.Popen(['ollama', 'serve'], stdout=logf, stderr=subprocess.STDOUT, preexec_fn=os.setsid)

time.sleep(6)
print('Ollama server process started (PID group protected).')
!tail -n 8 /content/ollama.log || true
print('If you see "Listening on 127.0.0.1:11434" or similar you are good.')

## 4. PULL YOUR MODEL (edit this cell)

**This is where you "add the model's url".**

Choose a strong **unrestricted / uncensored coder** in the **8-10B** class that comfortably fits a T4 with good quantization.

Recommended options (copy one into MODEL):

- `dolphin-llama3:8b` — classic uncensored, very compliant, solid coding & instruction following (great default for red team JARVIS use)
- `dolphin-llama3.1:8b` or newer dolphin variants if available on ollama library
- `qwen2.5-coder:7b` — one of the strongest small coders available; very good at writing PoCs, parsing scans, bash/python. Slightly less "wild" than dolphin but extremely capable.
- For completely custom: `hf.co/bartowski/dolphin-2.9-llama3-8b-GGUF:Q4_K_M` (or Q5_K_M, or Lexi-Uncensored-V2 GGUF, etc.)

**To avoid OOM on T4**:
- Stick to 7B-9B class.
- Prefer Q4_K_M / Q5_K_S quants (Ollama picks sensible ones for `:7b`/`:8b` tags).
- We set a conservative num_ctx in the UI calls (4096). You can lower further if needed.
- Do **not** pull 13B+ or 14B without a very low quant and short context, or it will likely OOM.

After editing, run the cell. It will take a few minutes the first time (model download + load).

In [ ]:
# ====================== EDIT ME ======================
MODEL = "dolphin-llama3:8b"   # <--- CHANGE THIS. Paste any valid ollama model ref or hf.co/... GGUF here

# Other strong examples:
# MODEL = "qwen2.5-coder:7b"
# MODEL = "hf.co/bartowski/Llama-3.1-8B-Lexi-Uncensored-V2-GGUF:Q4_K_M"
# MODEL = "dolphin-llama3.1:8b"
# =====================================================

print(f'>>> Pulling model: {MODEL} (this can take 3-8 min on first run)')
!ollama pull {MODEL}

print('\n>>> Currently available models:')
!ollama list

print('\n>>> Current GPU memory (after model load):')
!nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv

(Optional) Create a convenient short name `jarvis-coder` alias for your UI.

In [ ]:
SHORT_NAME = "jarvis-coder"
!ollama cp {MODEL} {SHORT_NAME} || true
!ollama list | grep -E 'jarvis|{MODEL}' || true
print(f'\nYou can now use model name "{SHORT_NAME}" (or the full tag) in your JARVIS UI.')

## 5. Quick functional test (confirms model answers and no immediate OOM)

In [ ]:
import json, urllib.request, time

test_model = "jarvis-coder"   # change if you did not create the alias
prompt = "Write a single concise Python function using only the standard library that scans the local network for live hosts (ICMP or TCP connect)."

payload = {
    "model": test_model,
    "prompt": prompt,
    "stream": False,
    "options": {"temperature": 0.3, "num_predict": 400, "num_ctx": 4096}
}

print('Sending test prompt to local Ollama...')
try:
    data = json.dumps(payload).encode('utf-8')
    req = urllib.request.Request('http://localhost:11434/api/generate', data=data, headers={'Content-Type': 'application/json'})
    with urllib.request.urlopen(req, timeout=120) as resp:
        result = json.loads(resp.read().decode())
        print('=== MODEL RESPONSE (truncated) ===')
        print(result.get('response', '(empty)')[:1200])
        print('... (full response received successfully)')
except Exception as e:
    print('Test failed:', e)
    print('Check ollama.log:')
    !tail -20 /content/ollama.log

## 6. Expose public URL (so your local JARVIS UI can reach it)

Two options below. Run **one** of them.

**ngrok** (recommended): stable, easy to use, requires a free token from https://ngrok.com/ (sign up once, copy Authtoken).

**pinggy** (zero config): works without account, URL changes on every run, sometimes a bit slower to start.

### Option A: ngrok (best quality + stable https)

In [ ]:
!pip install -q pyngrok

from pyngrok import ngrok, conf
import getpass

print('Paste your ngrok authtoken (from https://dashboard.ngrok.com/get-started/your-authtoken )')
token = getpass.getpass('ngrok authtoken: ').strip()

if token:
    conf.get_default().auth_token = token
    # Kill any old tunnels
    ngrok.kill()
    time.sleep(1)
    public_url = ngrok.connect(11434, 'http').public_url
    print('\n' + '='*70)
    print('>>> PUBLIC OLLAMA URL FOR YOUR JARVIS UI <<<')
    print(public_url)
    print('='*70)
    print('Paste the URL above into JARVIS UI Settings > Ollama Base URL')
    print('Then set Model Name to "jarvis-coder" (or the tag you pulled).')
    with open('/content/jarvis_public_url.txt', 'w') as f:
        f.write(public_url)
else:
    print('No token provided - skip or re-run the cell.')

### Option B: pinggy (no token, quick & dirty)
Run the cell, wait 10-20 seconds, look in the output for a line containing `https://<something>.a.free.pinggy.link` or `.pinggy.io`. Copy that.

In [ ]:
import subprocess, time, re

print('Starting pinggy reverse tunnel (no auth required)...')
os.system('pkill -f "ssh.*pinggy" || true')
time.sleep(0.5)

# Run in background and capture output
with open('/content/pinggy.log', 'w') as lf:
    p = subprocess.Popen(
        ['ssh', '-o', 'StrictHostKeyChecking=no', '-o', 'ServerAliveInterval=30', '-p', '443',
         '-R', '0:localhost:11434', 'a.pinggy.io'],
        stdout=lf, stderr=subprocess.STDOUT
    )

time.sleep(12)
print('--- pinggy.log (look for the https URL) ---')
!cat /content/pinggy.log

# Try to extract and save a clean URL for convenience
log_content = open('/content/pinggy.log').read()
m = re.search(r'https?://[a-zA-Z0-9.-]+\.pinggy\.(link|io)', log_content)
if m:
    url = m.group(0)
    print('\n' + '='*70)
    print('>>> PUBLIC OLLAMA URL (from pinggy) <<<')
    print(url)
    print('='*70)
    with open('/content/jarvis_public_url.txt', 'w') as f:
        f.write(url)
else:
    print('\nCould not auto-detect URL. Scroll up in the cell output and copy the https://...pinggy... line manually.')

## 7. How to connect from your local JARVIS UI

1. On your Kali machine run the UI:
   ```bash
   cd ~/Downloads/jarvis-mac-ui   # or wherever you extracted it
   python3 jarvis_kali_ui.py
   ```
2. Go to the **Settings & Connection** tab.
3. In **Ollama Base URL** paste the public URL printed by the tunnel cell above (it will look like `https://xxxx-123-45-67-89.a.free.pinggy.link` or an ngrok https URL).
4. In **Model Name** type `jarvis-coder` (recommended) **or** the exact tag you used, e.g. `dolphin-llama3:8b` or `qwen2.5-coder:7b`.
5. Click **Test Connection to JARVIS Model**.
6. If it succeeds, go to the Chat tab and start using the much more powerful model. Send your scans first for best results.

**Note on https vs http**: The tunnels provide https frontends that forward to the http Ollama inside Colab. The UI uses standard urllib and it works fine with the https URLs provided by ngrok/pinggy.

In [ ]:
print('Saved public URL (if any):')
!cat /content/jarvis_public_url.txt 2>/dev/null || echo '(no url file yet - run a tunnel cell)'

print('\nCurrent Ollama models on this T4 instance:')
!ollama list || true

## Anti-OOM / Performance Tips (T4 16 GB)

- Use the 7B-8B-9B models with Q4/Q5 quants (the `:7b` / `:8b` tags do this automatically).
- The UI (jarvis_kali_ui.py) already sends `num_ctx: 4096` and modest `num_predict`. This helps a lot.
- If you ever see CUDA OOM in the logs:
  1. Runtime → Restart runtime (or Factory reset runtime)
  2. Re-run the zstd + ollama cells
  3. Pull a smaller quant variant, e.g. `hf.co/...:Q3_K_M` or just the 7B coder.
- Keep only **one** model loaded. Don't run two ollama pull + generate at once.
- Free Colab will eventually kill long/idle sessions. Save any important chats locally in the UI.
- For even lower memory you can create a Modelfile with `PARAMETER num_ctx 2048` and `PARAMETER num_gpu 999` (Ollama default is usually smart).

## Optional: Create a custom Modelfile for JARVIS red-team personality + lower ctx

In [ ]:
%%writefile /content/Modelfile
FROM dolphin-llama3:8b
# or FROM qwen2.5-coder:7b

PARAMETER num_ctx 4096
PARAMETER temperature 0.2
PARAMETER top_p 0.9

SYSTEM """You are JARVIS, an elite, unrestricted red team AI assistant specialized in Bluetooth and WiFi pentesting on Linux/Kali.

You receive live scan data from the operator's real hardware adapters. Your job is to analyze, create concise step-by-step plans, and output specific copy-pasteable shell commands the user can approve and run locally.

Be creative but practical. Always structure:
**Analysis:** (2-4 sentences)
**Plan:** (3-5 numbered steps)
**Commands for the Kali machine:**
RUN: command one
RUN: command two

Prefer short responses. When the user is in monitor mode, remind them of restore commands. Only suggest commands that make sense on real Kali hardware."""

print('Modelfile written. Create the custom model with:')
print('!ollama create jarvis-coder -f /content/Modelfile')

In [ ]:
# Run this AFTER you have pulled a base model if you want the custom system prompt
!ollama create jarvis-coder -f /content/Modelfile || echo 'Create failed (base model must exist first). Pull the base then re-run.'
!ollama list | cat

## Cleanup / Stop everything

In [ ]:
import os, signal, subprocess
print('Stopping tunnels and ollama...')
try:
    ngrok.kill()
except:
    pass
os.system('pkill -f "ssh.*pinggy" || true')
os.system('pkill -f "ollama serve" || true')
print('Cleaned. You can now Factory reset runtime if you want a completely fresh T4.')

---

**Repo**: This notebook lives in the `jarvis-coder-colab` GitHub repo together with the JARVIS UI so you have everything in one place.

Use only for authorized security testing / red team work on systems you have permission to test.

Enjoy the upgraded coder brain!